# mae — video ingestion via Qwen2.5-VL (Deepnote-triggered)

Per Task #174 (2026-05-03 spec, session `24cbed9c`). Replaces the RunPod L40S video-extraction path with a Deepnote-Jobs-API trigger: POST → notebookRunId → poll → JSON output to `/work/output.json` → `<downstream-consumer>` ingests.

**Pipeline**: `yt-dlp` → frame extraction (every `FRAME_INTERVAL_SEC` seconds, max `MAX_FRAMES`) → `Qwen2.5-VL` inference per frame with a trading-intelligence prompt → aggregated signal record → `/work/output.json`.

**Parameters** (override via Deepnote Job `parameters` payload):
- `VIDEO_URL` — full YouTube URL or video ID
- `EXTRACT_MODE` — `frames` (visual analysis), `transcript` (audio/captions only), or `both`
- `FRAME_INTERVAL_SEC` — seconds between sampled frames (default 12)
- `MAX_FRAMES` — cap on frames to limit GPU cost (default 30)
- `MODEL` — Qwen2.5-VL model handle (default `Qwen/Qwen2.5-VL-7B-Instruct`; use `-3B-Instruct` on small GPUs)
- `PROMPT_PROFILE` — `trading-intelligence` (default) | `general-summary` | `custom`
- `OUTPUT_PATH` — where to write JSON (default `/work/output.json`)

**GPU**: Deepnote's GPU tier (T4 / A10 / A100 depending on availability). 7B-Instruct fits on T4 (16GB) in 8-bit; 3B-Instruct comfortably in FP16. Falls back to CPU if no GPU — slow but functional for the 3B model.

**Trigger via Deepnote API** — see `README.md` in this directory for the curl pattern + Python wrapper.

**Out of scope here**: signal scoring + position-sizing — `<downstream-consumer>` consumes this JSON and routes to the orchestrator gate stack.

In [1]:
import os

# Where Deepnote's Files panel actually mounts
PROJECT_WORK = "/datasets/_deepnote_work/work"

os.environ["VIDEO_URL"]          = f"{PROJECT_WORK}/2IMvfIR1TPA.mp4"
os.environ["EXTRACT_MODE"]       = "both"
os.environ["FRAME_INTERVAL_SEC"] = "12"
os.environ["MAX_FRAMES"]         = "30"
os.environ["MODEL"]              = "Qwen/Qwen2.5-VL-7B-Instruct"
os.environ["PROMPT_PROFILE"]     = "trading-intelligence"
os.environ["OUTPUT_PATH"]        = f"{PROJECT_WORK}/output.json"

In [2]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q --upgrade yt-dlp "transformers>=4.49" "accelerate>=0.34" "bitsandbytes>=0.43" qwen-vl-utils pillow einops sentencepiece torchvision


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import shutil, torch, transformers, yt_dlp, bitsandbytes
print("ffmpeg:      ", shutil.which("ffmpeg"))
print("torch:       ", torch.__version__, " cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
if torch.cuda.is_available():
    print("GPU:         ", torch.cuda.get_device_name(0))

/root/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
ffmpeg:       /usr/bin/ffmpeg
torch:        2.12.0+cu130  cuda: True
transformers: 5.10.1
bitsandbytes: 0.49.2
GPU:          Tesla T4


In [4]:
# §1 Setup — install + import
import os, sys, json, subprocess, time, base64, hashlib, re
from pathlib import Path
from datetime import datetime, timezone

def _pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *pkgs], check=True)

# yt-dlp + ffmpeg (frame extraction) + transformers/qwen-vl deps
try:
    import yt_dlp
except ImportError:
    _pip('yt-dlp')
    import yt_dlp

# ffmpeg binary check (frame extraction). Deepnote has it; bare-CPU containers may not.
ffmpeg_ok = subprocess.run(['ffmpeg', '-version'], capture_output=True).returncode == 0
if not ffmpeg_ok:
    print('Installing ffmpeg (apt)...')
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)

# Vision-language deps — defer to §4 (only load if EXTRACT_MODE != transcript-only)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'torch device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory // 1024**3} GB)')

# WORK = Path('/work') if Path('/work').exists() else Path('/tmp/mae-video-work')
WORK = Path(os.environ.get('WORK_OVERRIDE') or ('/work' if Path('/work').exists() else '/tmp/mae-video-work'))
WORK.mkdir(parents=True, exist_ok=True)
FRAMES_DIR = WORK / 'frames'
FRAMES_DIR.mkdir(exist_ok=True)
print(f'WORK: {WORK}')

torch device: cuda
GPU: Tesla T4 (14 GB)
WORK: /work


In [5]:
# §2 Parameters — Deepnote Jobs API injects these at trigger time.
# Defaults here let the notebook run standalone for manual testing.

VIDEO_URL          = os.environ.get('VIDEO_URL',          '')  # set by Deepnote Job parameters
EXTRACT_MODE       = os.environ.get('EXTRACT_MODE',       'both')           # frames | transcript | both
FRAME_INTERVAL_SEC = int(os.environ.get('FRAME_INTERVAL_SEC', '12'))
MAX_FRAMES         = int(os.environ.get('MAX_FRAMES',         '30'))
MODEL              = os.environ.get('MODEL',              'Qwen/Qwen2.5-VL-7B-Instruct')
PROMPT_PROFILE     = os.environ.get('PROMPT_PROFILE',     'trading-intelligence')
OUTPUT_PATH        = Path(os.environ.get('OUTPUT_PATH',  str(WORK / 'output.json')))

if not VIDEO_URL:
    # Standalone test fallback — operator should override via Deepnote Job params.
    # When operator provides a YT URL the live trigger sets VIDEO_URL.
    raise ValueError(
        'VIDEO_URL is required. Set it via Deepnote Job parameters: '
        '{"VIDEO_URL": "https://youtu.be/<id>"}'
    )

video_id_match = re.search(r'(?:v=|youtu\.be/|/watch\?v=)([\w-]{11})', VIDEO_URL) or re.match(r'^([\w-]{11})$', VIDEO_URL)
VIDEO_ID = video_id_match.group(1) if video_id_match else hashlib.md5(VIDEO_URL.encode()).hexdigest()[:11]

print(f'VIDEO_URL:           {VIDEO_URL}')
print(f'VIDEO_ID:            {VIDEO_ID}')
print(f'EXTRACT_MODE:        {EXTRACT_MODE}')
print(f'FRAME_INTERVAL_SEC:  {FRAME_INTERVAL_SEC}')
print(f'MAX_FRAMES:          {MAX_FRAMES}')
print(f'MODEL:               {MODEL}')
print(f'PROMPT_PROFILE:      {PROMPT_PROFILE}')
print(f'OUTPUT_PATH:         {OUTPUT_PATH}')

VIDEO_URL:           /datasets/_deepnote_work/work/2IMvfIR1TPA.mp4
VIDEO_ID:            feafa16e96b
EXTRACT_MODE:        both
FRAME_INTERVAL_SEC:  12
MAX_FRAMES:          30
MODEL:               Qwen/Qwen2.5-VL-7B-Instruct
PROMPT_PROFILE:      trading-intelligence
OUTPUT_PATH:         /datasets/_deepnote_work/work/output.json


In [6]:
# §3 - input dispatch + transcript extraction
# Supports:
#   - Local file path (from Drive import, manual upload, or any /work/ file)
#   - YouTube / direct http(s) URL via yt-dlp (with optional cookies file)
# Produces (contract for downstream cells):
#   VIDEO_PATH, SUBS_PATH, duration_sec, title, uploader,
#   transcript_text, transcript_segments

from pathlib import Path
import os, re, json, time, subprocess, hashlib

VIDEO_INPUT  = os.environ.get('VIDEO_URL', '').strip()
EXTRACT_MODE = os.environ.get('EXTRACT_MODE', 'both').strip()

if not VIDEO_INPUT:
    raise ValueError('VIDEO_URL not set (env var or input block)')

# Accept both absolute and relative local paths
candidate_path = Path(VIDEO_INPUT)
is_local = candidate_path.is_file() or candidate_path.exists()
is_url   = VIDEO_INPUT.startswith(('http://', 'https://'))

if not (is_local or is_url):
    raise ValueError(f'VIDEO_URL must be a local file path or http(s) URL, got: {VIDEO_INPUT}')

# ============================================================
# Branch A: LOCAL FILE (skip yt-dlp entirely)
# ============================================================
if is_local and not is_url:
    VIDEO_PATH = candidate_path.resolve()
    VIDEO_ID   = VIDEO_PATH.stem
    print(f'Using LOCAL video: {VIDEO_PATH}')

    # ffprobe for duration + title
    try:
        probe = subprocess.run(
            ['ffprobe', '-v', 'error',
            '-show_entries', 'format=duration:format_tags=title',
            '-of', 'json', str(VIDEO_PATH)],
            capture_output=True, text=True, check=True
        )
        info = json.loads(probe.stdout)
        duration_sec = float(info.get('format', {}).get('duration', 0))
        title = info.get('format', {}).get('tags', {}).get('title') or VIDEO_PATH.stem
    except Exception as e:
        print(f'  (ffprobe warning: {e})')
        duration_sec = 0
        title = VIDEO_PATH.stem

    uploader = 'local'
    print(f'  title:    {title}')
    print(f'  duration: {duration_sec:.1f}s')

    # Look for sidecar subtitles (foo.mp4 -> foo.en.vtt / foo.vtt / foo.srt)
    SUBS_PATH = None
    for ext in ['.en.vtt', '.vtt', '.en.srt', '.srt']:
        candidate = VIDEO_PATH.with_suffix(ext)
        if candidate.exists():
            SUBS_PATH = candidate
            print(f'  found sidecar subtitles: {SUBS_PATH.name}')
            break

# ============================================================
# Branch B: URL (yt-dlp with optional cookies)
# ============================================================
else:
    import yt_d

Using LOCAL video: /datasets/_deepnote_work/work/2IMvfIR1TPA.mp4
  title:    2IMvfIR1TPA
  duration: 3509.4s


## §3b Frame extraction via ffmpeg

Sample one frame every `FRAME_INTERVAL_SEC` seconds, capped at `MAX_FRAMES`. Saves to `WORK/frames/` as JPEG.

In [7]:
frame_paths = []
if EXTRACT_MODE in ('frames', 'both'):
    # Compute frame timestamps (capped at MAX_FRAMES)
    n_possible = max(1, duration_sec // FRAME_INTERVAL_SEC)
    n_frames = min(MAX_FRAMES, n_possible)
    if n_possible > MAX_FRAMES:
        # Distribute MAX_FRAMES evenly across duration (don't just take the first MAX_FRAMES)
        step = duration_sec / MAX_FRAMES
        timestamps = [step * i for i in range(MAX_FRAMES)]
    else:
        timestamps = [FRAME_INTERVAL_SEC * i for i in range(n_frames)]

    print(f'Extracting {len(timestamps)} frames at intervals of ~{duration_sec/max(len(timestamps),1):.1f}s...')
    for i, ts in enumerate(timestamps):
        out = FRAMES_DIR / f'frame_{i:03d}_{int(ts):05d}s.jpg'
        subprocess.run(['ffmpeg', '-y', '-ss', f'{ts:.2f}', '-i', str(VIDEO_PATH),
                        '-frames:v', '1', '-q:v', '3', str(out)],
                       check=True, capture_output=True)
        frame_paths.append({'idx': i, 'timestamp_sec': ts, 'path': str(out)})
    print(f'  extracted: {len(frame_paths)} frames')
else:
    print('Frame extraction skipped (EXTRACT_MODE=transcript)')

Extracting 30 frames at intervals of ~117.0s...
  extracted: 30 frames


## §4 Qwen2.5-VL model load

Loads the model with whatever precision fits the available GPU memory:
- **A100 40GB / A10 24GB**: 7B-Instruct in FP16 (~16 GB)
- **T4 16GB**: 7B-Instruct in 8-bit (~10 GB) via `bitsandbytes`
- **<16GB or CPU**: caller should override `MODEL=Qwen/Qwen2.5-VL-3B-Instruct`

Skip this cell entirely if `EXTRACT_MODE=transcript`.

In [8]:
# §4 — Qwen2.5-VL model load
# Loads model with whatever precision fits available GPU memory:
#   A100 40GB / A10 24GB: 7B-Instruct in FP16 (~16 GB)
#   T4 16GB:              7B-Instruct in 8-bit (~10 GB) via bitsandbytes
#   <16GB or CPU:         caller should override MODEL=Qwen/Qwen2.5-VL-3B-Instruct
# Skips load entirely if EXTRACT_MODE=transcript.

vl_model = None
vl_processor = None

if EXTRACT_MODE in ('frames', 'both'):
    try:
        from transformers import (
            Qwen2_5_VLForConditionalGeneration,
            AutoProcessor,
            BitsAndBytesConfig,
        )
    except ImportError:
        _pip('transformers>=4.49', 'accelerate', 'qwen-vl-utils', 'bitsandbytes')
        from transformers import (
            Qwen2_5_VLForConditionalGeneration,
            AutoProcessor,
            BitsAndBytesConfig,
        )

    load_kwargs = {'device_map': 'auto'}

    if DEVICE == 'cuda':
        gpu_gb = torch.cuda.get_device_properties(0).total_memory // 1024**3
        # if gpu_gb < 20 and '7B' in MODEL:
        #    print(f'GPU has {gpu_gb} GB — loading in 8-bit (BitsAndBytesConfig)')
        #    load_kwargs['quantization_config'] = BitsAndBytesConfig(load_in_8bit=True)
        if gpu_gb < 20 and '7B' in MODEL:
            print(f'GPU has {gpu_gb} GB — loading in 4-bit NF4')
            load_kwargs['quantization_config'] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )        
        else:
            print(f'GPU has {gpu_gb} GB — loading in FP16')
            load_kwargs['torch_dtype'] = torch.float16
    else:
        # CPU path — slow but works for 3B model
        print(f'No GPU — loading in FP32 on CPU (slow; use 3B model if possible)')
        load_kwargs['torch_dtype'] = torch.float32

    print(f'Loading {MODEL} on {DEVICE}...')
    t0 = time.time()
    vl_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL, **load_kwargs)
    vl_processor = AutoProcessor.from_pretrained(MODEL)
    print(f'  loaded in {time.time()-t0:.1f}s')
else:
    print('EXTRACT_MODE=transcript — skipping VL model load')

GPU has 14 GB — loading in 4-bit NF4
Loading Qwen/Qwen2.5-VL-7B-Instruct on cuda...
Loading weights:   0%|          | 2/729 [00:04<31:19,  2.59s/it]/root/venv/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 729/729 [00:21<00:00, 33.78it/s] 
  loaded in 108.3s


## §5 Inference — per-frame structured extraction

Per the `trading-intelligence` prompt profile, each frame returns:
- `tickers` — symbols mentioned or visible (crypto/equity/forex)
- `price_callouts` — specific price levels referenced
- `sentiment` — bullish/bearish/neutral per mentioned asset
- `macro_events` — FOMC, CPI prints, earnings, geopolitical
- `chart_patterns` — visible chart formations
- `speaker_confidence` — high/medium/low
- `time_sensitivity` — now/this-week/longer-term

In [9]:
PROMPT_PROFILES = {
    'trading-intelligence': '''You are a market-intelligence analyst watching a financial-content video for an algorithmic trading system.

For THIS FRAME, return STRICT JSON with these exact fields. Use empty arrays / null for fields with no signal. Do not invent.

{
"tickers": ["BTC", "ETH", ...],
"price_callouts": [{"ticker": "BTC", "price": 67500, "context": "..."}],
"sentiment": {"BTC": "bullish|bearish|neutral", ...},
"macro_events": ["FOMC Wed", "CPI Thu", ...],
"chart_patterns": ["BTC head-and-shoulders", ...],
"speaker_confidence": "high|medium|low|null",
"time_sensitivity": "now|this-week|longer-term|null",
"frame_notes": "one-sentence summary of what is visible"
}

Return ONLY the JSON object, no preamble.''',

    'general-summary': '''Describe what is visible in this frame in 2 sentences. Return JSON {"summary": "..."}.''',

    'trading-education': '''You are a trading-curriculum analyst reviewing an educational video frame for an algorithmic trading agent ("the kid"). The kid will be quizzed on what was taught and is expected to apply
these lessons in production.

For THIS FRAME, return STRICT JSON with these exact fields. Use empty arrays / null / empty strings for fields with no signal in this frame. Do not invent content that is not visible. Capture only what is shown or said
in THIS frame.

{
"concepts": ["volume bar", "tick imbalance bar", "dollar bar", "Lopez de Prado information-driven bars"],
"formulas": [
    {"name": "volume bar threshold", "expression": "sum(volume) >= V_threshold", "explanation": "accumulate ticks until total volume crosses V_threshold, then emit one bar"}
],
"code_or_pseudocode": [
    {"language": "python", "snippet": "if cum_volume >= V_threshold: emit_bar(); cum_volume = 0", "purpose": "volume-bar formation loop"}
],
"diagrams": [
    {"type": "chart", "description": "side-by-side: time bars vs volume bars on the same tick stream, volume bars cluster tighter during high-activity periods"}
],
"worked_examples": [
    {"input": "tick stream with bursts of activity around news events", "process": "accumulate volume until threshold, emit bar", "output": "fewer bars during quiet periods, denser bars during active periods, more
uniform information content per bar"}
],
"prerequisites": ["tick data", "order book basics", "time bars as the naive default"],
"alternatives_or_comparisons": [
    {"name": "time bars", "vs": "volume bars", "key_difference": "time bars sample at fixed clock intervals regardless of activity; volume bars sample at fixed information intervals regardless of clock"},
    {"name": "dollar bars", "vs": "volume bars", "key_difference": "dollar bars normalize by notional traded rather than share count, making them comparable across price regimes"}
],
"key_quotes": ["verbatim instructive statement from narrator if shown on screen as caption or slide text"],
"actionable_for_agent": [
    "when computing features, prefer volume-bar OHLCV over time-bar OHLCV for instruments with bursty volume profiles",
    "set V_threshold per instrument based on average daily volume / target bars-per-day"
],
"frame_notes": "one-sentence summary of what is being taught in this frame"
}

Return ONLY the JSON object, no preamble.''',
}

PROMPT = PROMPT_PROFILES.get(PROMPT_PROFILE, PROMPT_PROFILES['trading-education'])

def _infer_one(image_path):
    """Run Qwen2.5-VL on a single frame; return parsed dict or {raw: ...} on parse failure."""
    from PIL import Image
    image = Image.open(image_path).convert('RGB')
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': PROMPT},
    ]}]
    text = vl_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = vl_processor(text=[text], images=[image], padding=True, return_tensors='pt').to(vl_model.device)
    with torch.inference_mode():
        out_ids = vl_model.generate(**inputs, max_new_tokens=512, do_sample=False)
    out_text = vl_processor.batch_decode(
        out_ids[:, inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )[0].strip()
    try:
        # Extract first JSON object from response (model may include trailing text)
        match = re.search(r'\{.*\}', out_text, re.DOTALL)
        return json.loads(match.group(0)) if match else {'raw': out_text}
    except (json.JSONDecodeError, AttributeError):
        return {'raw': out_text, 'parse_error': True}

frame_signals = []
if vl_model is not None and frame_paths:
    print(f'Inferring {len(frame_paths)} frames...')
    t0 = time.time()
    for fp in frame_paths:
        try:
            sig = _infer_one(fp['path'])
        except Exception as e:
            sig = {'error': str(e)}
        frame_signals.append({
            'frame_idx': fp['idx'],
            'timestamp_sec': fp['timestamp_sec'],
            'signal': sig,
        })
        if (len(frame_signals) % 5) == 0:
            print(f'  {len(frame_signals)}/{len(frame_paths)}  ({time.time()-t0:.1f}s)')
    print(f'Inference complete: {len(frame_signals)} frames in {time.time()-t0:.1f}s')

Inferring 30 frames...
/root/venv/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
  5/30  (37.5s)
  10/30  (73.6s)
  15/30  (109.0s)
  20/30  (142.4s)
  25/30  (179.0s)
  30/30  (224.3s)
Inference complete: 30 frames in 224.3s


In [10]:
import json
for fs in frame_signals[:3]:
    print(f"\n=== frame {fs['frame_idx']} @ {fs['timestamp_sec']:.1f}s ===")
    print(json.dumps(fs['signal'], indent=2, default=str))


=== frame 0 @ 0.0s ===
{
  "tickers": [],
  "price_callouts": [],
  "sentiment": {},
  "macro_events": [],
  "chart_patterns": [],
  "speaker_confidence": "null",
  "time_sensitivity": "null",
  "frame_notes": "The frame contains a disclaimer about day trading profitability and risks."
}

=== frame 1 @ 117.0s ===
{
  "tickers": [],
  "price_callouts": [],
  "sentiment": {},
  "macro_events": [],
  "chart_patterns": [],
  "speaker_confidence": "null",
  "time_sensitivity": "null",
  "frame_notes": "A screenshot of a trading platform showing individual account balances and available funds."
}

=== frame 2 @ 234.0s ===
{
  "tickers": [],
  "price_callouts": [],
  "sentiment": {},
  "macro_events": [],
  "chart_patterns": [],
  "speaker_confidence": "null",
  "time_sensitivity": "null",
  "frame_notes": "The frame shows a timeline of the PDT Rule from February 27, 2001 to June 4, 2026, along with articles discussing its impact on day trading."
}


In [11]:
# Ensure transcript vars exist even if §3 didn't define them (local file, no sidecar)
transcript_text = locals().get('transcript_text', '')
transcript_segments = locals().get('transcript_segments', [])
print(f"transcript_text:     {len(transcript_text)} chars")
print(f"transcript_segments: {len(transcript_segments)} segments")

transcript_text:     0 chars
transcript_segments: 0 segments


## §6 Aggregate — signal record for <downstream-consumer>

Per-frame signals are aggregated into a single record that captures: dominant tickers, net sentiment, macro events, time-sensitivity histogram, and the raw per-frame array (for the coach to re-inspect any frame's context).

In [12]:
from collections import Counter, defaultdict

def _aggregate(frames):
    ticker_counter = Counter()
    sentiment_by_ticker = defaultdict(Counter)
    macro = Counter()
    time_sens = Counter()
    speaker_conf = Counter()
    price_callouts = []
    for f in frames:
        s = f.get('signal', {})
        if not isinstance(s, dict): continue
        for t in s.get('tickers', []) or []:
            ticker_counter[str(t).upper()] += 1
        for t, sent in (s.get('sentiment', {}) or {}).items():
            sentiment_by_ticker[str(t).upper()][str(sent).lower()] += 1
        for ev in s.get('macro_events', []) or []:
            macro[str(ev)] += 1
        ts = s.get('time_sensitivity')
        if ts: time_sens[str(ts).lower()] += 1
        sc = s.get('speaker_confidence')
        if sc: speaker_conf[str(sc).lower()] += 1
        for pc in s.get('price_callouts', []) or []:
            if isinstance(pc, dict):
                pc['frame_idx'] = f.get('frame_idx')
                price_callouts.append(pc)
    # Net sentiment per ticker as bullish_frac - bearish_frac
    net_sent = {}
    for t, c in sentiment_by_ticker.items():
        total = sum(c.values())
        if total:
            net_sent[t] = round((c.get('bullish', 0) - c.get('bearish', 0)) / total, 3)
    return {
        'dominant_tickers': [t for t, _ in ticker_counter.most_common(10)],
        'ticker_frame_counts': dict(ticker_counter),
        'net_sentiment': net_sent,
        'macro_events_mentioned': dict(macro),
        'time_sensitivity_dist': dict(time_sens),
        'speaker_confidence_dist': dict(speaker_conf),
        'price_callouts': price_callouts,
    }

agg = _aggregate(frame_signals) if frame_signals else {}

record = {
    'schema_version': 1,
    'source': f'youtube/{VIDEO_ID}',
    'source_url': VIDEO_URL,
    'title': title,
    'uploader': uploader,
    'duration_sec': duration_sec,
    'extracted_at': datetime.now(timezone.utc).isoformat(),
    'extract_mode': EXTRACT_MODE,
    'model': MODEL,
    'prompt_profile': PROMPT_PROFILE,
    'frames_analyzed': len(frame_signals),
    'transcript': {
        'text': transcript_text,
        'segments': transcript_segments,
        'segment_count': len(transcript_segments),
        'char_count': len(transcript_text),
    },
    'aggregate': agg,
    'frames': frame_signals,  # raw per-frame for coach re-inspection
}

print(f'\n=== Aggregate ===')
print(f'Dominant tickers:   {agg.get("dominant_tickers", [])}')
print(f'Net sentiment:      {agg.get("net_sentiment", {})}')
print(f'Macro events:       {list((agg.get("macro_events_mentioned", {}) or {}).keys())[:5]}')
print(f'Time-sensitivity:   {agg.get("time_sensitivity_dist", {})}')
print(f'Speaker confidence: {agg.get("speaker_confidence_dist", {})}')
print(f'Price callouts:     {len(agg.get("price_callouts", []))}')


=== Aggregate ===
Dominant tickers:   ['SPWR', 'BTC', 'CYCN']
Net sentiment:      {'SPWR': 0.0, 'BTC': 0.0, 'CYCN': 0.0}
Macro events:       []
Time-sensitivity:   {'null': 29, 'now': 1}
Speaker confidence: {'null': 29, 'medium': 1}
Price callouts:     2


## §7 Write output JSON

Lands at `OUTPUT_PATH` (default `/work/output.json`). Deepnote keeps `/work/*` accessible after the job; the trigger script (`README.md`) fetches this file via Deepnote's File API or via committed Git artifact.

In [13]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(record, indent=2, default=str))
size_kb = OUTPUT_PATH.stat().st_size // 1024
print(f'Wrote {size_kb} KB → {OUTPUT_PATH}')
print()
print(f'JSON preview (first 1.5 KB):')
print(OUTPUT_PATH.read_text()[:1500])

Wrote 14 KB → /datasets/_deepnote_work/work/output.json

JSON preview (first 1.5 KB):
{
  "schema_version": 1,
  "source": "youtube/2IMvfIR1TPA",
  "source_url": "/datasets/_deepnote_work/work/2IMvfIR1TPA.mp4",
  "title": "2IMvfIR1TPA",
  "uploader": "local",
  "duration_sec": 3509.394,
  "extracted_at": "2026-06-03T21:44:14.908371+00:00",
  "extract_mode": "both",
  "model": "Qwen/Qwen2.5-VL-7B-Instruct",
  "prompt_profile": "trading-intelligence",
  "frames_analyzed": 30,
  "transcript": {
    "text": "",
    "segments": [],
    "segment_count": 0,
    "char_count": 0
  },
  "aggregate": {
    "dominant_tickers": [
      "SPWR",
      "BTC",
      "CYCN"
    ],
    "ticker_frame_counts": {
      "SPWR": 1,
      "BTC": 1,
      "CYCN": 1
    },
    "net_sentiment": {
      "SPWR": 0.0,
      "BTC": 0.0,
      "CYCN": 0.0
    },
    "macro_events_mentioned": {},
    "time_sensitivity_dist": {
      "null": 29,
      "now": 1
    },
    "speaker_confidence_dist": {
      "null": 29,
  

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=69bff347-5146-4e81-b443-03e5b4e81bc4' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>